# VoxBind - Sample Metrics Dashboard

Summarises the cached `metrics.json` files written for every sampled pocket
under `voxbind/exps/<experiment>/samples/<run>/target_*/`.

### How to use
1. Run **section 0 (Setup)** once.
2. **Section 1** lists every experiment that has generated samples - review it.
3. In **section 2**, set `SELECTED` to an `idx` (or experiment name) from that list.
4. Run **sections 3-5** for the selected experiment; **section 6** compares all of them.

### Notes
- This dashboard only *reads* pre-computed `metrics.json` (it does not run
  RDKit or docking). Refresh those files with the Streamlit app
  (`notebook/webapp/app.py`).
- **Docking metrics (Vina) are optional.** They come from a separate evaluation
  step and may not be computed yet - targets without them show `NaN` in the
  `vina_*` columns and are flagged. Every other metric still prints normally.
- `status` column: `fresh` = metrics newer than `samples.sdf` -
  `stale` = out of date - `no-metrics` = not evaluated yet -
  `no-samples` = empty target directory.

In [1]:
# Section 0 - Setup: helper functions (run once).
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 100)


def show(obj, round_to=3):
    """Render a DataFrame/Series in Jupyter, falling back to plain text."""
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        obj = obj.round(round_to)
    try:
        from IPython.display import display
        display(obj)
    except Exception:
        print(obj)


# -- locate the repository ---------------------------------------------------
def find_repo_root() -> Path:
    """Walk up from the working directory until voxbind/exps/ is found."""
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "voxbind" / "exps").is_dir():
            return cand
    fallback = Path("/home/shpark/prj-denovo/VoxBind")
    if (fallback / "voxbind" / "exps").is_dir():
        return fallback
    raise FileNotFoundError("Could not find voxbind/exps/ - run from inside the repo.")


REPO = find_repo_root()
EXPS_ROOT = REPO / "voxbind" / "exps"
METRICS_VERSION = 2          # version stamp written by notebook/webapp/metrics.py
print(f"repo      : {REPO}")
print(f"exps root : {EXPS_ROOT}")


# -- experiment / run / target discovery -------------------------------------
def list_runs(exp_dir: Path) -> list:
    """Sampling-run subdirs under <exp>/samples/ that hold target_* folders."""
    samples = exp_dir / "samples"
    if not samples.is_dir():
        return []
    return sorted(d.name for d in samples.iterdir()
                  if d.is_dir() and any(d.glob("target_*")))


def list_experiments() -> list:
    """Experiment names with at least one non-empty sampling run."""
    if not EXPS_ROOT.is_dir():
        return []
    return sorted(p.name for p in EXPS_ROOT.iterdir()
                  if p.is_dir() and list_runs(p))


def list_targets(exp_name: str, run: str) -> list:
    """Sorted target_* directories for one (experiment, run)."""
    run_dir = EXPS_ROOT / exp_name / "samples" / run
    if not run_dir.is_dir():
        return []
    return sorted(t for t in run_dir.glob("target_*") if t.is_dir())


# -- metrics.json loading and freshness --------------------------------------
def metrics_status(target_dir: Path) -> str:
    """One of: fresh | stale | no-metrics | no-samples."""
    mp = target_dir / "metrics.json"
    sdf = target_dir / "samples.sdf"
    if not sdf.exists():
        return "no-samples"
    if not mp.exists():
        return "no-metrics"
    try:
        data = json.loads(mp.read_text())
    except (json.JSONDecodeError, OSError):
        return "no-metrics"
    if data.get("version") != METRICS_VERSION:
        return "stale"
    return "fresh" if mp.stat().st_mtime >= sdf.stat().st_mtime else "stale"


def load_metrics(target_dir: Path):
    """Parsed metrics.json dict, or None if absent / unreadable."""
    mp = target_dir / "metrics.json"
    if not mp.exists():
        return None
    try:
        return json.loads(mp.read_text())
    except (json.JSONDecodeError, OSError):
        return None


# -- docking metrics (Vina) - optional, may not be computed yet --------------
# Per-sample docking is expected either as a nested {"vina": {...}} dict
# (poc_evaluate.py convention) or as flat vina_* keys. When nothing is found,
# the vina_* columns stay NaN and the dashboard flags the target/experiment.
VINA_COLS = ["vina_score", "vina_min", "vina_dock"]
_VINA_NESTED = {"score_only": "vina_score", "minimize": "vina_min", "dock": "vina_dock"}


def sample_docking(sample: dict) -> dict:
    """Extract {vina_score, vina_min, vina_dock} from one sample record."""
    out = {}
    v = sample.get("vina")
    if isinstance(v, dict) and "error" not in v:
        for src, dst in _VINA_NESTED.items():
            if isinstance(v.get(src), (int, float)):
                out[dst] = float(v[src])
    for col in VINA_COLS:
        if isinstance(sample.get(col), (int, float)):
            out[col] = float(sample[col])
    return out


def target_docking(metrics: dict) -> dict:
    """Mean Vina metrics for a target - NaN for any not computed."""
    out = {c: float("nan") for c in VINA_COLS}
    acc = {c: [] for c in VINA_COLS}
    for s in metrics.get("samples", []):
        for c, val in sample_docking(s).items():
            acc[c].append(val)
    for c in VINA_COLS:
        if acc[c]:
            out[c] = float(np.mean(acc[c]))
    # fall back to aggregate-level keys, in case a future pipeline writes them there
    agg = metrics.get("aggregates", {})
    aliases = {
        "vina_score": ("vina_score_mean", "vina_score", "Vina score mean"),
        "vina_min":   ("vina_min_mean", "vina_min", "Vina min mean"),
        "vina_dock":  ("vina_dock_mean", "vina_dock", "Vina dock mean"),
    }
    for c, names in aliases.items():
        if math.isnan(out[c]):
            for n in names:
                if isinstance(agg.get(n), (int, float)):
                    out[c] = float(agg[n])
                    break
    return out


def has_docking(metrics) -> bool:
    """True if any Vina metric is present in this target's metrics."""
    return metrics is not None and any(
        not math.isnan(v) for v in target_docking(metrics).values()
    )


def n_vina_failed(metrics) -> int:
    """Number of samples whose vina recorded an error (toolchain crash, worker
    fault, etc.). Distinguishes 'docking ran and failed on every sample' from
    'docking never ran' - both leave vina_* NaN, but the first is recoverable
    by re-running. Reads `aggregates.vina_n_failed` when present (written by
    metrics.py >= 2026-05-21); falls back to counting samples whose `vina`
    dict carries an `error` key for older caches."""
    if not metrics:
        return 0
    agg = metrics.get("aggregates", {})
    n = agg.get("vina_n_failed")
    if isinstance(n, int):
        return n
    return sum(1 for s in metrics.get("samples", [])
               if isinstance(s.get("vina"), dict) and "error" in s["vina"])


# -- table builders ----------------------------------------------------------
PER_TARGET_COLS = [
    "target", "status", "computed_at", "n_total", "n_valid",
    "validity", "uniqueness", "diversity",
    "qed", "sa", "logp", "lipinski", "clash_free",
] + VINA_COLS + ["vina_n_failed", "high_affinity"]


def per_target_table(exp_name: str, run: str) -> pd.DataFrame:
    """One summary row per target_* directory of (experiment, run)."""
    rows = []
    for tdir in list_targets(exp_name, run):
        row = {"target": tdir.name, "status": metrics_status(tdir)}
        m = load_metrics(tdir)
        if m is not None:
            agg = m.get("aggregates", {})
            ca = m.get("computed_at", "") or ""
            row["computed_at"] = ca[:16].replace("T", " ")
            row["n_total"] = agg.get("n_total")
            row["n_valid"] = agg.get("n_valid")
            row["validity"] = agg.get("validity")
            row["uniqueness"] = agg.get("uniqueness")
            row["diversity"] = agg.get("diversity")
            row["qed"] = agg.get("qed_mean")
            row["sa"] = agg.get("sa_mean")
            row["logp"] = agg.get("logp_mean")
            row["lipinski"] = agg.get("lipinski_mean")
            row["high_affinity"] = agg.get("high_affinity")
            row["vina_n_failed"] = n_vina_failed(m)
            samples = m.get("samples", [])
            if samples:
                row["clash_free"] = float(np.mean([
                    s.get("interactions", {}).get("n_clashes", 0) == 0
                    for s in samples
                ]))
            row.update(target_docking(m))
        rows.append(row)
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    for c in PER_TARGET_COLS:
        if c not in df.columns:
            df[c] = np.nan
    return df[PER_TARGET_COLS]


def experiment_aggregate(df: pd.DataFrame) -> dict:
    """Mean of the per-target table over targets (NaN where not computed).
    `vina_n_failed` is summed (not averaged) - it's a total sample-error count
    across the run, not a per-target rate."""
    numeric = ["validity", "uniqueness", "diversity", "qed", "sa", "logp",
               "lipinski", "clash_free", "high_affinity"] + VINA_COLS
    out = {"n_targets": int(len(df))}
    out["evaluated"] = int((df["status"] == "fresh").sum()) if "status" in df else 0
    for c in numeric:
        out[c] = float(df[c].mean(skipna=True)) if (c in df and len(df)) else float("nan")
    out["vina_n_failed"] = (int(df["vina_n_failed"].fillna(0).sum())
                            if ("vina_n_failed" in df and len(df)) else 0)
    return out


def per_sample_table(exp_name: str, run: str, target_name: str) -> pd.DataFrame:
    """Every generated molecule for one target (empty if not evaluated).
    `vina_status` makes docking failures visible: OK = docked successfully,
    ERR = ran but crashed (full traceback summary in `vina_err`), — = never
    docked. Lets a reader distinguish 'NaN because docking errored' from
    'NaN because docking wasn't requested'."""
    tdir = EXPS_ROOT / exp_name / "samples" / run / target_name
    m = load_metrics(tdir)
    if m is None:
        return pd.DataFrame()
    rows = []
    for i, s in enumerate(m.get("samples", [])):
        ix = s.get("interactions", {})
        row = {
            "idx": i,
            "smiles": s.get("smiles", ""),
            "n_atoms": s.get("n_atoms"),
            "qed": s.get("qed"),
            "sa": s.get("sa"),
            "logp": s.get("logp"),
            "lipinski": s.get("lipinski"),
            "n_contacts": ix.get("n_contacts"),
            "n_clashes": ix.get("n_clashes"),
            "min_dist": ix.get("min_dist"),
        }
        for c in VINA_COLS:
            row[c] = np.nan
        v = s.get("vina")
        if isinstance(v, dict) and "error" in v:
            row["vina_status"] = "ERR"
            row["vina_err"] = v.get("error", "")
        elif isinstance(v, dict):
            row["vina_status"] = "OK"
            row["vina_err"] = ""
        else:
            row["vina_status"] = "—"
            row["vina_err"] = ""
        row.update(sample_docking(s))
        rows.append(row)
    return pd.DataFrame(rows)


def first_evaluated(exp_name: str):
    """Return (run, target_name) of the fresh-metrics target with most samples.

    Falls back to the first run/target if none have fresh metrics.
    """
    runs = list_runs(EXPS_ROOT / exp_name)
    best = None  # (n_valid, run, target_name)
    for run in runs:
        for t in list_targets(exp_name, run):
            if metrics_status(t) != "fresh":
                continue
            m = load_metrics(t) or {}
            n = m.get("aggregates", {}).get("n_valid", 0) or 0
            if best is None or n > best[0]:
                best = (n, run, t.name)
    if best is not None:
        return best[1], best[2]
    if runs:
        ts = list_targets(exp_name, runs[0])
        return runs[0], (ts[0].name if ts else None)
    return None, None


print("helpers ready: list_experiments, list_runs, list_targets,")
print("               per_target_table, experiment_aggregate, per_sample_table,")
print("               n_vina_failed (sample-error count per target)")

repo      : /home/shpark/prj-denovo/VoxBind
exps root : /home/shpark/prj-denovo/VoxBind/voxbind/exps
helpers ready: list_experiments, list_runs, list_targets,
               per_target_table, experiment_aggregate, per_sample_table,
               n_vina_failed (sample-error count per target)


## Experiment list

`EXPS` is auto-discovered: every experiment under `voxbind/exps/` that has at
least one non-empty sampling run. To curate which experiments the cross-
experiment comparison (section 6) reports on, replace it with a hand-picked
list, e.g. `EXPS = ["260514_voxbind_100ep340", "260515_voxbind_100ep_density"]`.

The `metrics` / `docking` columns are coverage counts (`computed / total`).

In [2]:
# Section 1 - list every experiment that has generated samples.
EXPS = list_experiments()

overview = []
for i, exp_name in enumerate(EXPS):
    runs = list_runs(EXPS_ROOT / exp_name)
    n_targets = n_fresh = n_dock = n_failed = 0
    for run in runs:
        for tdir in list_targets(exp_name, run):
            n_targets += 1
            m = load_metrics(tdir)
            if metrics_status(tdir) == "fresh":
                n_fresh += 1
            if has_docking(m):
                n_dock += 1
            n_failed += n_vina_failed(m)
    # docking column: target-level coverage + (if any) sample-level error count.
    # Distinguishes 'never computed' from 'computed but every dock crashed'.
    if n_dock:
        dock_str = f"{n_dock}/{n_targets}"
    elif n_failed:
        dock_str = f"0/{n_targets}"
    else:
        dock_str = "not computed"
    if n_failed:
        dock_str += f"  ·  {n_failed} errored"
    overview.append({
        "idx": i,
        "experiment": exp_name,
        "runs": ", ".join(runs),
        "targets": n_targets,
        "metrics": f"{n_fresh}/{n_targets}",
        "docking": dock_str,
    })

if not EXPS:
    print(f"No experiments with generated samples under {EXPS_ROOT}")
else:
    print(f"{len(EXPS)} experiment(s) with generated samples:")
    show(pd.DataFrame(overview).set_index("idx"))


# Section 6 - compare every experiment in EXPS.
def target_reference(metrics):
    """Reference-ligand metrics for one target, or {} when absent.

    Read from the target's metrics.json `reference` key (qed/sa/logp/lipinski
    and the Vina scores). validity/uniqueness/diversity/clash_free are not
    defined for a single ligand.
    """
    ref = (metrics or {}).get("reference")
    if not isinstance(ref, dict) or "error" in ref:
        return {}
    out = {}
    for k in ("qed", "sa", "logp", "lipinski"):
        if isinstance(ref.get(k), (int, float)):
            out[k] = float(ref[k])
    v = ref.get("vina")
    if isinstance(v, dict) and "error" not in v:
        for src, dst in _VINA_NESTED.items():
            if isinstance(v.get(src), (int, float)):
                out[dst] = float(v[src])
    return out


def experiment_reference_row(exp_name):
    """Mean reference-ligand metrics over every target of an experiment.

    Gathered from each target's metrics.json - there is no `reference/` run on
    disk. Returned with the same keys as experiment_aggregate(); the set-level
    columns (validity/uniqueness/diversity/clash_free) stay NaN.
    """
    refs = []
    for run in list_runs(EXPS_ROOT / exp_name):
        for tdir in list_targets(exp_name, run):
            r = target_reference(load_metrics(tdir))
            if r:
                refs.append(r)
    metric_cols = ["qed", "sa", "logp", "lipinski"] + VINA_COLS
    row = {c: float("nan") for c in
           ["validity", "uniqueness", "diversity", "clash_free"] + metric_cols}
    row["n_targets"] = row["evaluated"] = len(refs)
    for c in metric_cols:
        vals = [r[c] for r in refs if c in r]
        if vals:
            row[c] = float(np.mean(vals))
    return row


def _highlight_best(df):
    """Comparison-table CSS: underline the best run within each experiment,
    bold the best across all experiments. Higher is better, except the
    lower-is-better Vina columns; logp and the count columns are not ranked.
    The 'reference' rows are a baseline and are excluded from the ranking.
    """
    higher = ["validity", "uniqueness", "diversity", "qed", "sa",
              "lipinski", "clash_free", "high_affinity"]
    lower = ["vina_score", "vina_min", "vina_dock"]
    css = pd.DataFrame("", index=df.index, columns=df.columns)
    experiments = df.index.get_level_values("experiment")
    not_ref = pd.Series(df.index.get_level_values("run") != "reference",
                        index=df.index)
    for col in higher + lower:
        if col not in df.columns:
            continue
        series = df[col].where(not_ref)
        is_max = col in higher
        best = series.max() if is_max else series.min()
        if pd.notna(best):
            css.loc[series == best, col] += "font-weight: bold;"
        for exp in experiments.unique():
            grp = series[experiments == exp].dropna()
            if grp.empty:
                continue
            ebest = grp.max() if is_max else grp.min()
            in_exp = pd.Series(experiments == exp, index=df.index)
            css.loc[in_exp & (series == ebest), col] += "text-decoration: underline;"
    return css


def _color_vs_reference(df):
    """Comparison-table CSS: shade every non-reference run by how its metric
    compares with the reference ligand of the *same* experiment - light green
    when the run beats the reference, light red when it is worse. Columns with
    no reference value (validity/uniqueness/diversity/clash_free) or no defined
    direction (logp, the count columns) are left uncoloured.
    """
    higher = ["qed", "sa", "lipinski"]               # higher is better
    lower = ["vina_score", "vina_min", "vina_dock"]  # lower is better
    green = "background-color: #d8f0d8;"
    red = "background-color: #f5d8d8;"
    css = pd.DataFrame("", index=df.index, columns=df.columns)
    experiments = df.index.get_level_values("experiment")
    runs = df.index.get_level_values("run")
    for exp in experiments.unique():
        ref_rows = df[(experiments == exp) & (runs == "reference")]
        if ref_rows.empty:
            continue
        ref = ref_rows.iloc[0]
        run_idx = df.index[(experiments == exp) & (runs != "reference")]
        for col in higher + lower:
            if col not in df.columns or pd.isna(ref[col]):
                continue
            better_when_higher = col in higher
            for idx in run_idx:
                val = df.at[idx, col]
                if pd.isna(val):
                    continue
                better = val > ref[col] if better_when_higher else val < ref[col]
                css.at[idx, col] = green if better else red
    return css


def _separator_table_styles(df):
    """Return set_table_styles entries drawing a thick black border above the
    first row of each experiment block (after the first one). With a MultiIndex,
    the 'experiment' th uses rowspan and only exists on the *first* row of its
    block - putting border-top there guarantees the line spans from the very
    first column (experiment) across the entire row.
    """
    experiments = df.index.get_level_values("experiment")
    styles = []
    for i in range(1, len(df)):
        if experiments[i] != experiments[i - 1]:
            # nth-child is 1-based; selector matches both th (index) and td cells
            styles.append({
                "selector": (f"tbody tr:nth-child({i + 1}) th, "
                             f"tbody tr:nth-child({i + 1}) td"),
                "props": "border-top: 2px solid black;",
            })
    return styles


cmp_rows = []
for exp_name in EXPS:
    for run in list_runs(EXPS_ROOT / exp_name):
        df = per_target_table(exp_name, run)
        if df.empty:
            continue
        cmp_rows.append({"experiment": exp_name, "run": run,
                         **experiment_aggregate(df)})
    # gather the reference ligands into a virtual 'reference' run (no folder)
    ref_row = experiment_reference_row(exp_name)
    if ref_row["n_targets"]:
        cmp_rows.append({"experiment": exp_name, "run": "reference", **ref_row})

cmp_df = pd.DataFrame(cmp_rows)
if cmp_df.empty:
    print("Nothing to compare - no experiments with samples.")
else:
    _cmp = cmp_df.set_index(["experiment", "run"])
    _styled = (_cmp.style
               .apply(_highlight_best, axis=None)
               .apply(_color_vs_reference, axis=None)
               .set_table_styles(_separator_table_styles(_cmp), overwrite=False)
               .format(precision=3, na_rep="NaN"))
    show(_styled)
    print("  underline = best within an experiment  ·  bold = best across all")
    print("  green / red cell = run metric better / worse than its reference ligand")
    print("  'reference' run = mean reference-ligand metrics (baseline, not ranked)")
    missing = [c for c in VINA_COLS if cmp_df[c].isna().all()]
    if missing:
        print(f"  docking not computed anywhere - NaN columns: {', '.join(missing)}")
    n_err_total = int(cmp_df["vina_n_failed"].fillna(0).sum()) if "vina_n_failed" in cmp_df else 0
    if n_err_total:
        print(f"  vina_n_failed column = total sample-level dock errors per (experiment, run); {n_err_total} across all runs")

12 experiment(s) with generated samples:


,experiment,runs,targets,metrics,docking
idx,,,,,
0,260514_voxbind_100ep_original,"res_ep7999_test10, res_ep7999_val",20,20/20,20/20
1,260515_voxbind_100ep_density,"res_ep7999_test10, res_ep7999_val",20,20/20,20/20
2,260516_voxbind_100ep_noise,"res_ep7999_test10, res_ep7999_val",20,20/20,19/20
3,260518_voxbind_10k_baseline,res_ep99_test,92,92/92,92/92
4,260518_voxbind_10k_density,res_ep99_test,66,65/66,65/66
5,260518_voxbind_10k_noise,res_ep99_test,63,62/63,62/63
6,260519_voxbind_10k_density_aligned,"res_test_n10, res_test_n10_ep138",58,58/58,57/58
7,260521_voxbind_10k_density_mae_frozen,"res_ep199_quick10, res_ep99_quick10",4,4/4,4/4
8,260523_voxbind_10k_density_vit_mae_frozen_e0199_snap,"res_ep199_10pockets, res_ep199_quick10",12,12/12,12/12


  underline = best within an experiment  ·  bold = best across all
  green / red cell = run metric better / worse than its reference ligand
  'reference' run = mean reference-ligand metrics (baseline, not ranked)


## Results

Set `SELECTED` below to an `idx` from the table in section 1 (or to an
experiment-name string), then run the remaining cells.

### Experiments - sample directories

In [3]:
# ======================================================================
#  vvv  SELECT THE EXPERIMENT  vvv   - idx from section 1, or its name
SELECTED = 0
# ======================================================================

if not EXPS:
    raise RuntimeError("No experiments to select - see section 1.")
if isinstance(SELECTED, int):
    sel_exp = EXPS[SELECTED]
elif SELECTED in EXPS:
    sel_exp = SELECTED
else:
    raise ValueError(f"SELECTED={SELECTED!r} is not a valid idx or name; "
                     f"choose from {EXPS}")

sel_runs = list_runs(EXPS_ROOT / sel_exp)
print(f"selected experiment : {sel_exp}")
print(f"sampling runs       : {sel_runs}")

agg_rows = []
for run in sel_runs:
    df = per_target_table(sel_exp, run)
    if df.empty:
        continue
    agg_rows.append({"run": run, **experiment_aggregate(df)})

if agg_rows:
    agg_df = pd.DataFrame(agg_rows).set_index("run")
    print(f"{sel_exp} - aggregate metrics (mean over targets)")
    show(agg_df)
    if agg_df[VINA_COLS].isna().all().all():
        print("  docking metrics (vina_*): not computed for this experiment")
else:
    print(f"{sel_exp}: no targets to aggregate.")

selected experiment : 260514_voxbind_100ep_original
sampling runs       : ['res_ep7999_test10', 'res_ep7999_val']
260514_voxbind_100ep_original - aggregate metrics (mean over targets)


,n_targets,evaluated,validity,uniqueness,diversity,qed,sa,logp,lipinski,clash_free,high_affinity,vina_score,vina_min,vina_dock,vina_n_failed
run,,,,,,,,,,,,,,,
res_ep7999_test10,10,10,1.0,1.0,0.752,0.484,0.595,2.01,4.75,1.0,0.47,-5.096,-6.440,-6.936,0
res_ep7999_val,10,10,1.0,1.0,0.727,0.527,0.577,2.14,4.78,1.0,NaN,-6.688,-7.494,-8.388,0


### Experiments - over runs

In [4]:
# Section 3 - per-target metrics for every run of the selected experiment.
def _color_targets_vs_reference(df, exp_name, run):
    """Per-target table CSS: shade each metric green/red when the target's
    generated molecules beat / fall short of that target's own reference
    ligand (from its metrics.json). qed/sa/lipinski are higher-is-better,
    vina_* lower-is-better; columns with no reference value or no defined
    direction (logp, the count columns) stay uncoloured.
    """
    higher = ["qed", "sa", "lipinski"]               # higher is better
    lower = ["vina_score", "vina_min", "vina_dock"]  # lower is better
    green = "background-color: #d8f0d8;"
    red = "background-color: #f5d8d8;"
    css = pd.DataFrame("", index=df.index, columns=df.columns)
    for tname in df.index:
        tdir = EXPS_ROOT / exp_name / "samples" / run / tname
        ref = target_reference(load_metrics(tdir))
        if not ref:
            continue
        for col in higher + lower:
            if col not in df.columns or col not in ref:
                continue
            val = df.at[tname, col]
            if pd.isna(val):
                continue
            better = val > ref[col] if col in higher else val < ref[col]
            css.at[tname, col] = green if better else red
    return css


for run in sel_runs:
    df = per_target_table(sel_exp, run)
    print(f"\n{'='*72}\n{sel_exp}  /  {run}\n{'='*72}")
    if df.empty:
        print("  (no target directories)")
        continue
    df = df.set_index("target")
    styled = (df.style
              .apply(_color_targets_vs_reference, exp_name=sel_exp, run=run,
                     axis=None)
              .format(precision=3, na_rep="NaN"))
    show(styled)
    print("  green / red cell = target metric better / worse than its reference ligand")
    n_no_dock = int(df[VINA_COLS].isna().all(axis=1).sum())
    if n_no_dock == len(df):
        print("  docking (vina_*): not computed for any target in this run")
    elif n_no_dock:
        print(f"  docking (vina_*): not computed for {n_no_dock}/{len(df)} targets")

if not sel_runs:
    print(f"{sel_exp}: no sampling runs.")


260514_voxbind_100ep_original  /  res_ep7999_test10


,status,computed_at,n_total,n_valid,validity,uniqueness,diversity,qed,sa,logp,lipinski,clash_free,vina_score,vina_min,vina_dock,vina_n_failed,high_affinity
target,,,,,,,,,,,,,,,,,
target_00,fresh,2026-05-19 22:00,10,10,1.000,1.000,0.849,0.468,0.631,0.876,5.000,1.000,2.221,-5.676,-6.942,0,0.100
target_01,fresh,2026-05-18 06:30,10,10,1.000,1.000,0.740,0.459,0.593,3.025,4.700,1.000,-5.679,-5.926,-6.734,0,0.100
target_02,fresh,2026-05-18 06:31,10,10,1.000,1.000,0.656,0.507,0.525,2.661,4.600,1.000,-6.965,-7.446,-8.137,0,1.000
target_03,fresh,2026-05-18 06:31,10,10,1.000,1.000,0.901,0.478,0.675,0.953,5.000,1.000,-3.307,-4.037,-5.138,0,0.100
target_04,fresh,2026-05-18 06:32,10,10,1.000,1.000,0.719,0.545,0.573,2.013,4.500,1.000,-8.896,-9.534,-10.361,0,0.700
target_05,fresh,2026-05-18 06:32,10,10,1.000,1.000,0.906,0.472,0.723,1.196,5.000,1.000,-1.612,-2.059,1.034,0,0.000
target_07,fresh,2026-05-18 06:35,10,10,1.000,1.000,0.602,0.505,0.514,2.219,4.700,1.000,-7.955,-8.604,-9.287,0,0.500
target_08,fresh,2026-05-18 06:36,10,10,1.000,1.000,0.674,0.433,0.544,3.239,4.400,1.000,-7.597,-7.911,-8.839,0,1.000
target_09,fresh,2026-05-18 06:36,10,10,1.000,1.000,0.679,0.420,0.530,2.014,4.800,1.000,-4.257,-5.697,-6.587,0,0.900


  green / red cell = target metric better / worse than its reference ligand

260514_voxbind_100ep_original  /  res_ep7999_val


,status,computed_at,n_total,n_valid,validity,uniqueness,diversity,qed,sa,logp,lipinski,clash_free,vina_score,vina_min,vina_dock,vina_n_failed,high_affinity
target,,,,,,,,,,,,,,,,,
target_00,fresh,2026-05-17 22:06,10,10,1.000,1.000,0.704,0.458,0.526,1.838,4.600,1.000,-2.403,-4.689,-7.629,0,NaN
target_01,fresh,2026-05-17 22:06,10,10,1.000,1.000,0.769,0.545,0.636,0.931,5.000,1.000,-6.077,-6.909,-7.192,0,NaN
target_02,fresh,2026-05-17 22:08,10,10,1.000,1.000,0.724,0.548,0.573,3.238,4.900,1.000,-8.017,-8.521,-9.384,0,NaN
target_03,fresh,2026-05-17 22:10,10,10,1.000,1.000,0.744,0.521,0.566,2.961,4.900,1.000,-6.339,-6.708,-8.039,0,NaN
target_04,fresh,2026-05-17 22:12,10,10,1.000,1.000,0.716,0.420,0.533,1.620,4.700,1.000,-7.736,-8.311,-9.016,0,NaN
target_05,fresh,2026-05-17 22:14,10,10,1.000,1.000,0.701,0.528,0.594,3.832,4.600,1.000,-7.891,-8.631,-9.023,0,NaN
target_06,fresh,2026-05-17 22:15,10,10,1.000,1.000,0.726,0.635,0.636,2.159,4.900,1.000,-7.141,-7.895,-8.204,0,NaN
target_07,fresh,2026-05-17 22:18,10,10,1.000,1.000,0.694,0.423,0.535,1.889,4.300,1.000,-7.301,-7.896,-8.781,0,NaN
target_08,fresh,2026-05-17 22:18,10,10,1.000,1.000,0.777,0.618,0.619,1.953,4.900,1.000,-6.495,-7.155,-7.765,0,NaN


  green / red cell = target metric better / worse than its reference ligand


### Experiments - per sample

Every generated molecule for a single target. Leave `RUN` / `SELECTED_TARGET`
as `None` to auto-pick the evaluated target with the most samples, or set them
to an index or name to inspect a specific one.

In [5]:
# Section 5 - per-sample metrics for one target.
RUN = None              # None -> auto | index into sel_runs | run-name string
SELECTED_TARGET = None  # None -> auto | index into the run's targets | target name

if not sel_runs:
    print(f"{sel_exp}: no sampling runs.")
else:
    if RUN is None and SELECTED_TARGET is None:
        run, tname = first_evaluated(sel_exp)
    else:
        run = sel_runs[RUN] if isinstance(RUN, int) else (RUN or sel_runs[0])
        tnames = [t.name for t in list_targets(sel_exp, run)]
        if SELECTED_TARGET is None:
            tname = tnames[0] if tnames else None
        elif isinstance(SELECTED_TARGET, int):
            tname = tnames[SELECTED_TARGET]
        else:
            tname = SELECTED_TARGET

    if run is None or tname is None:
        print(f"{sel_exp}: no samples to display.")
    else:
        tdir = EXPS_ROOT / sel_exp / "samples" / run / tname
        df = per_sample_table(sel_exp, run, tname)
        print(f"{sel_exp}  /  {run}  /  {tname}   "
              f"(status: {metrics_status(tdir)})")
        if df.empty:
            print("  no metrics.json - this target has not been evaluated yet")
        else:
            show(df.set_index("idx"))
            if df[VINA_COLS].isna().all().all():
                print("  docking (vina_*): not computed for these samples")

260514_voxbind_100ep_original  /  res_ep7999_test10  /  target_00   (status: fresh)


,smiles,n_atoms,qed,sa,logp,lipinski,n_contacts,n_clashes,min_dist,vina_score,vina_min,vina_dock,vina_status,vina_err
idx,,,,,,,,,,,,,,
0,CC=NC1=CC(=O)N=C2N(C1)C[C@]2(N)[C@@H]1CCCN(C2=CC[C@H](C)[C@H](C)O2)C1,28,0.751,0.51,1.911,5,25,0,2.787,67.094,-6.134,-8.847,OK,
1,CC[C@@H](N)CO[C@@H](CCNCO)COC,15,0.347,0.69,-0.315,5,15,0,2.786,-4.055,-3.733,-5.894,OK,
2,NCNCN[C@H]1C=C[C@H](OC(=O)c2ccc3c(c2)CCC[C@H]3N)C=C1,25,0.267,0.66,1.096,5,24,0,2.800,-6.464,-6.884,-7.792,OK,
3,N=Cc1ccc2c(c1)N=C1CN=CC=C1O2,16,0.710,0.68,2.117,5,16,0,2.826,-6.072,-6.660,-7.362,OK,
4,C=C[C@H]1C=C(C[C@H](O)[C@@H](N)C=O)CC(OC)=C1,17,0.532,0.55,0.926,5,17,0,2.550,-5.427,-6.082,-7.055,OK,
5,CC(=CC(N)=O)COC(=O)O,11,0.451,0.78,0.113,5,11,0,3.016,-4.934,-5.293,-5.695,OK,
6,CCC(=N)CCC[C@@H](O)[C@H](C)COCNC,16,0.304,0.65,1.777,5,15,0,2.818,-4.713,-5.070,-6.713,OK,
7,CC=C1C=C(O[C@H]2C[C@@H]3CCCC(C=CNN=CN)=C3O2)C(O)=CC1,25,0.404,0.52,3.485,5,22,0,3.000,-5.155,-7.915,-7.777,OK,
8,OCCCNC1=N[C@H]2OCNCN2CC1,15,0.519,0.63,-1.119,5,14,0,2.867,-3.812,-4.544,-6.464,OK,
